# 01 — Data Audit

This notebook checks whether the current processed queue-time dataset is usable for analysis.

The goal is not to generate final insights yet. The goal is to understand coverage, missing values, attraction representation and whether we have enough snapshots to support stronger EDA and modeling.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from parkflow.data.data_quality import (
    add_audit_time_columns,
    build_coverage_summary,
    hourly_coverage_report,
    load_best_available_dataset,
    missingness_report,
    ride_coverage_report,
)


## 1. Load the best available processed dataset


In [ ]:
df, path = load_best_available_dataset()
path, df.shape


In [ ]:
df = add_audit_time_columns(df)
df.head()


## 2. High-level coverage summary


In [ ]:
summary = build_coverage_summary(df)
pd.DataFrame(summary.items(), columns=['metric', 'value'])


## 3. Coverage by attraction

This table helps us identify whether some attractions are underrepresented, always closed or producing unusual wait-time values.


In [ ]:
ride_report = ride_coverage_report(df)
ride_report.head(30)


## 4. Coverage by date and hour


In [ ]:
hourly = hourly_coverage_report(df)
hourly.head(30)


In [ ]:
if not hourly.empty and 'snapshots' in hourly.columns:
    coverage_pivot = hourly.pivot_table(
        index='audit_date_local',
        columns='audit_hour_local',
        values='snapshots',
        aggfunc='sum',
        fill_value=0,
    )
    display(coverage_pivot)


## 5. Missingness report


In [ ]:
missingness_report(df)


## 6. Initial decision checklist

Use this checklist after each collection cycle:

- Do we have more than one snapshot?
- Do we have multiple hours covered?
- Do we have more than one day covered?
- Are the timestamps aligned with local park time?
- Are most attractions represented consistently?
- Are wait times mostly zero because the park was closed, or because data is sparse?
- Is there enough coverage to make EDA claims without overinterpreting the data?
